1: Instalação de Dependências

In [0]:
%pip install transformers torch tqdm

2: Imports e Configurações

In [0]:
import os
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import pipeline

# Configuração de caminhos usando Volumes do Unity Catalog
# Note que para Pandas acessar o Volume, usamos o caminho direto do sistema de arquivos
input_path = '/Volumes/workspace/voc/voc/voc_gold_2025.csv'
output_path = '/Volumes/workspace/voc/voc/voc_gold_processed_2025.csv'

3: Carregamento dos Dados

In [0]:
if not os.path.exists(input_path):
    raise FileNotFoundError(f"O arquivo não foi encontrado no Volume: {input_path}")

df = pd.read_csv(input_path)

# Limpeza inicial
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['origem'] = df['origem'].astype(str).str.strip()
df['usuario'] = df['usuario'].astype(str).str.strip()
df['mensagem'] = df['mensagem'].astype(str).fillna('')

print(f"Dados carregados: {df.shape[0]} linhas.")
df.head()

Inicialização do Modelo (Hugging Face) - Voce pode usar aqui sua API

In [0]:
use_hf = True
sentiment_pipe = None

try:
    print("Carregando modelo de sentimento (isso pode levar alguns minutos na primeira vez)...")
    sentiment_pipe = pipeline(
        task='sentiment-analysis',
        model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
        top_k=None,
        device=-1 # Força CPU para evitar erros em clusters sem GPU
    )
    print("Modelo carregado com sucesso!")
except Exception as err:
    use_hf = False
    print(f"Erro ao carregar modelo, usando fallback de regras. Erro: {err}")

5: Funções de Lógica (Sentimento e Categoria)

In [0]:
POS_WORDS = ['bom','boa','otimo','ótimo','excelente','perfeito','obrigado','obrigada','show','resolvido','funcionou','sucesso','rapido','rápido']
NEG_WORDS = ['erro','bug','falha','caiu','fora do ar','instavel','instável','lento','lenta','demora','sem retorno','sem resposta','nao funciona','não funciona','problema','travando','trava','pessimo','péssimo','horrivel','horrível','inaceitavel','inaceitável','urgente','401','500','timeout','rejeitado','recusado']

def rule_sentiment(text_val):
    t = str(text_val).lower()
    pos_hits = sum([1 for w in POS_WORDS if w in t])
    neg_hits = sum([1 for w in NEG_WORDS if w in t])
    if neg_hits > pos_hits and neg_hits > 0: return 'NEGATIVA'
    if pos_hits > neg_hits and pos_hits > 0: return 'POSITIVA'
    return 'NEUTRA'

CAT_RULES = [
    ('INSTABILIDADE NA PLATAFORMA', ['fora do ar','instavel','instável','caiu','travando','trava','lento','lenta','timeout','latencia','latência','500','502','503']),
    ('API/AUTENTICACAO', ['api','token','401','403','autenticacao','autenticação','oauth','jwt']),
    ('BUGS', ['bug','erro','falha','stack','excecao','exceção','quebrou']),
    ('SUPORTE/ATENDIMENTO', ['sem retorno','sem resposta','demora','atraso','chamado','ticket','suporte','atendimento']),
    ('FINANCEIRO', ['fatura','cobranca','cobrança','boleto','pix','nota fiscal','nf','reembolso','pagamento','preco','preço','plano']),
    ('TREINAMENTO/ONBOARDING', ['treinamento','trilha','onboarding','curso','como usar','tutorial','documentacao','documentação']),
    ('FEATURE REQUEST', ['seria bom','poderia ter','faltou','queria','gostaria','sugestao','sugestão','melhoria','feature'])
]

def categorize(text_val):
    t = str(text_val).lower()
    for cat, kws in CAT_RULES:
        for kw in kws:
            if kw in t: return cat
    return 'FEEDBACK GERAL'

def hf_to_pt(label_val):
    mapping = {'LABEL_0': 'NEGATIVA', 'LABEL_1': 'NEUTRA', 'LABEL_2': 'POSITIVA'}
    return mapping.get(label_val, 'NEUTRA')

6: Execução do Processamento

In [0]:
texts = df['mensagem'].astype(str).tolist()
sent_out = []

if sentiment_pipe is not None:
    batch_size = 32 # Reduzido para estabilidade no Community Edition
    for start_idx in tqdm(range(0, len(texts), batch_size), desc="Classificando Sentimentos"):
        batch_txt = texts[start_idx:start_idx+batch_size]
        preds = sentiment_pipe(batch_txt)
        for item in preds:
            # Pega o label com maior score
            best = sorted(item, key=lambda x: x['score'], reverse=True)[0]
            sent_out.append(hf_to_pt(best['label']))
else:
    for t in tqdm(texts, desc="Processando via Regras"):
        sent_out.append(rule_sentiment(t))

df['sentimentalidade'] = sent_out
df['categoria'] = [categorize(x) for x in tqdm(texts, desc="Categorizando")]

print("Processamento concluído!")

7: Salvamento no Volume

In [0]:
df.to_csv(output_path, index=False)
print(f"Sucesso! O arquivo processado foi salvo em: {output_path}")